##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")
x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224, interpolation="bilinear"),
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)
], name="cifar10_mobilenetv2")

model.summary()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1),
]

history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print("MobileNetV2 (frozen) test accuracy:", test_acc)
print("MobileNetV2 (frozen) test loss    :", test_loss)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

print("Trainable layers in backbone:", sum(l.trainable for l in base_model.layers), "/", len(base_model.layers))

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_ft = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

test_loss_ft, test_acc_ft = model.evaluate(x_test, y_test, verbose=0)
print("MobileNetV2 (fine-tuned) test accuracy:", test_acc_ft)
print("MobileNetV2 (fine-tuned) test loss    :", test_loss_ft)

Model: "cifar10_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing_1 (Resizing)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 1059s 1s/step - accuracy: 0.6810 - loss: 0.9101 - val_accuracy: 0.8118 - val_loss: 0.5494 - learning_rate: 0.0010
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 860s 1s/step - accuracy: 0.7490 - loss: 0.7245 - val_accuracy: 0.8198 - val_loss: 0.5309 - learning_rate: 0.0010
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 925s 1s/step - accuracy: 0.7606 - loss: 0.6890 - val_accuracy: 0.8266 - val_loss: 0.5136 - learning_rate: 0.0010
MobileNetV2 (frozen) test accuracy: 0.8127999901771545
MobileNetV2 (frozen) test loss    : 0.5444945693016052
Trainable layers in backbone: 30 / 154
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 1072s 2s/step - accuracy: 0.7170 - loss: 0.8174 - val_accuracy: 0.8292 - val_loss: 0.5025
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 1021s 1s/step - accuracy: 0.7776 - loss: 0.6393 - val_accuracy: 0.8406 - val_loss: 0.4591
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 987s 1s/step - accuracy: 0.7988 - loss: 0.5812 - val_accuracy: 0.8534 - val_loss: 0.4128
MobileNe

- Which model achieved the highest accuracy?
The fine-tuned MobileNetV2 achieved the highest test accuracy at 85.14%, compared to the frozen model's 81.28%. Unfreezing the last 30 layers and training with a lower learning rate (1e-5) allowed the model to adapt its learned features to CIFAR-10, gaining roughly 4% improvement.


- Which model trained faster?
The frozen model trained faster. Each epoch took around 860–1059 seconds with only 12,810 trainable parameters (the classification head), while the fine-tuned model took 987–1072 seconds per epoch because it had to compute and update gradients for 30 additional backbone layers. Fewer trainable parameters means less computation per step.


- How might the architecture explain the differences?
MobileNetV2 was pre-trained on ImageNet at 224×224 resolution, so its convolutional filters encode rich general-purpose features (edges, textures, shapes). When frozen, those features are used as-is effective, but not optimized for CIFAR-10's 32×32 images and 10-class structure. Fine-tuning the last 30 layers lets the deeper, more task-specific filters adjust to the domain gap (low-res inputs upscaled to 224×224, different class granularity), which is why accuracy improves. The depthwise separable convolutions in MobileNetV2 keep the fine-tuning cost relatively low despite unfreezing many layers, but there's still a noticeable speed difference versus training only a small dense head.

